# NLP Preprocessing Pipeline
**End-to-end pipeline:** Kaggle download → EDA → Text Cleaning → BERT Tokenization → Stratified Split → Export

---

## 0. Setup & Dependencies

In [ ]:
# Install required packages (run once)
# !pip install kaggle transformers datasets scikit-learn pandas matplotlib seaborn wordcloud

In [ ]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import kagglehub
import pandas as pd

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('✅ All libraries imported successfully')

## 1. Configuration

In [ ]:
# ─── USER CONFIG ──────────────────────────────────────────────────────────────
KAGGLE_DATASET   = 'nn_final/sentiment140'   # <-- change to your dataset slug
TEXT_COL         = 'statement'                    # column with raw text
LABEL_COL        = 'status'               # column with class labels
RANDOM_STATE     = 42

DATA_DIR         = Path('data')
OUTPUT_DIR       = Path('output')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)



## 2. Download Dataset from Kaggle

In [ ]:
# Download latest version
path = kagglehub.dataset_download("suchintikasarkar/sentiment-analysis-for-mental-health")

files = os.listdir(path=path)

df = pd.read_csv(os.path.join(path, files[0]))

In [ ]:
print(f'Shape : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Normalise to (text, label) if needed ──────────────────────────────────────
# Map numeric labels → readable strings  (edit to match your dataset)
LABEL_MAP = {0: 'negative', 2: 'neutral', 4: 'positive'}

df = df[[TEXT_COL, LABEL_COL]].copy()
if df[LABEL_COL].dtype != object:
    df[LABEL_COL] = df[LABEL_COL].map(LABEL_MAP).fillna(df[LABEL_COL].astype(str))

print(f'Working dataframe: {df.shape}')
print(f'Label values     : {sorted(df[LABEL_COL].unique())}')
df.head(3)

## 3. Exploratory Data Analysis

### 3.1 Class Distribution

In [ ]:
label_counts = df[LABEL_COL].value_counts().sort_index()
label_pct    = label_counts / len(df) * 100

print('Class distribution')
print('─' * 38)
for cls, cnt in label_counts.items():
    bar = '█' * int(label_pct[cls] / 2)
    print(f'{str(cls):>12}  {cnt:>8,}  ({label_pct[cls]:5.1f}%)  {bar}')
print('─' * 38)
print(f'{"TOTAL":>12}  {len(df):>8,}  (100.0%)')

# Imbalance ratio
imbalance = label_counts.max() / label_counts.min()
print(f'\nImbalance ratio (max/min): {imbalance:.2f}x')
if imbalance > 2:
    print('⚠  Consider class-weighting or oversampling during training.')

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5),                     # slightly wider overall figure
    gridspec_kw={'width_ratios': [1, 1.3]}  # give pie chart more space
)

# Bar chart
axes[0].bar(
    label_counts.index.astype(str),
    label_counts.values,
    color=sns.color_palette('muted', len(label_counts)),
    edgecolor='white'
)

axes[0].set_title('Class Counts', fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count (Thousands)')
axes[0].tick_params(axis='x', rotation=30)
for label in axes[0].get_xticklabels():
    label.set_ha('right')

axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}')
)

for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + len(df)*0.003,
        f'{bar.get_height()}',
        ha='center',
        va='bottom',
        fontsize=9
    )

wedges, texts, autotexts = axes[1].pie(
    label_counts.values,
    labels=label_counts.index.astype(str),
    autopct='%1.1f%%',
    startangle=140,
    radius=1.15,  
    textprops={'fontsize': 11},
    colors=sns.color_palette('muted', len(label_counts))
)

# Make percentage labels larger + bold
for autotext in autotexts:
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')

axes[1].set_title('Class Proportions', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', bbox_inches='tight')
plt.show()

### 3.2 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %':     missing_pct.round(4)
})
print('Missing value audit')
print(missing_df.to_string())

empty_text = (df[TEXT_COL].astype(str).str.strip() == '').sum()
print(f'\nEmpty / whitespace-only text rows: {empty_text:,}')

In [ ]:
# Drop rows with missing text or label
before = len(df)
df.dropna(subset=[TEXT_COL, LABEL_COL], inplace=True)
df = df[df[TEXT_COL].astype(str).str.strip() != ''].copy()
df.reset_index(drop=True, inplace=True)
print(f'Rows removed (missing): {before - len(df):,}  |  Remaining: {len(df):,}')

### 3.3 Duplicate Detection

In [ ]:
n_exact_dup    = df.duplicated(subset=[TEXT_COL]).sum()
n_full_dup     = df.duplicated().sum()
n_cross_label  = df[df.duplicated(subset=[TEXT_COL], keep=False)] \
                   .groupby(TEXT_COL)[LABEL_COL].nunique()
n_conflict     = (n_cross_label > 1).sum()

print(f'Duplicate text rows (same text, any label) : {n_exact_dup:,}')
print(f'Fully duplicate rows (text + label)        : {n_full_dup:,}')
print(f'Conflicting label rows (same text, diff lbl): {n_conflict:,}')

if n_conflict > 0:
    print('\n⚠  Conflicting labels — showing top 5 examples:')
    conflict_texts = n_cross_label[n_cross_label > 1].index[:5]
    print(df[df[TEXT_COL].isin(conflict_texts)][[TEXT_COL, LABEL_COL]].to_string())

In [ ]:
# Remove exact duplicates (keep first occurrence)
before = len(df)
df.drop_duplicates(subset=[TEXT_COL], keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Rows removed (duplicates): {before - len(df):,}  |  Remaining: {len(df):,}')

### 3.4 Text Length Statistics

In [ ]:
df['char_len']  = df[TEXT_COL].str.len()
df['word_len']  = df[TEXT_COL].str.split().str.len()

stats = df[['char_len', 'word_len']].describe(percentiles=[.25, .5, .75, .90, .95, .99])
print('Text length statistics')
print(stats.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Character length — overall
axes[0, 0].hist(df['char_len'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0, 0].axvline(df['char_len'].median(), color='tomato', lw=1.8, label=f'Median: {df["char_len"].median():.0f}')
axes[0, 0].set_title('Character Length Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Characters'); axes[0, 0].set_ylabel('Count'); axes[0, 0].legend()

# Word length — overall
axes[0, 1].hist(df['word_len'], bins=40, color='mediumseagreen', edgecolor='white', alpha=0.85)
axes[0, 1].axvline(df['word_len'].median(), color='tomato', lw=1.8, label=f'Median: {df["word_len"].median():.0f}')
axes[0, 1].set_title('Word Length Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Words'); axes[0, 1].set_ylabel('Count'); axes[0, 1].legend()

# Character length per class
# Character length by class — grouped bar chart
char_stats = df.groupby(LABEL_COL)['char_len'].agg(['mean', 'median'])
classes = sorted(df[LABEL_COL].unique())
colors = sns.color_palette('muted', len(classes))

x = np.arange(len(classes))
width = 0.35

# Mean bars
axes[1, 0].bar(
    x - width/2,
    char_stats.loc[classes, 'mean'],
    width,
    label='Mean',
    color=colors,
    alpha=0.85
)

# Median bars (slightly darker)
axes[1, 0].bar(
    x + width/2,
    char_stats.loc[classes, 'median'],
    width,
    label='Median',
    color=colors,
    alpha=0.55
)

axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels([str(c) for c in classes], rotation=25, ha='right')
axes[1, 0].set_title('Character Length by Class', fontweight='bold')
axes[1, 0].set_xlabel('Class')
axes[1, 0].set_ylabel('Characters')
axes[1, 0].legend()



# Word length per class — box plot
classes = df[LABEL_COL].unique()
data_per_class = [df.loc[df[LABEL_COL] == cls, 'word_len'].values for cls in sorted(classes)]
bp = axes[1, 1].boxplot(
    data_per_class,
    labels=[str(c) for c in sorted(classes)],
    patch_artist=True,
    notch=True,
    showfliers=True
)

colors = sns.color_palette('muted', len(classes))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Keep box readable but still show most outliers
upper_limit = df['word_len'].quantile(0.98)
axes[1, 1].set_ylim(0, upper_limit)

axes[1, 1].set_title('Word Length by Class (Box Plot)', fontweight='bold')
axes[1, 1].set_xlabel('Class')
axes[1, 1].set_ylabel('Words')

plt.suptitle('Text Length Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'text_length_analysis.png', bbox_inches='tight')
plt.show()
print('Figure saved → output/text_length_analysis.png')

## 4. Text Cleaning

In [ ]:
# ─── Compiled patterns (define once for speed) ────────────────────────────────
_RE_URL     = re.compile(r'https?://\S+|www\.\S+')
_RE_MENTION = re.compile(r'@\w+')
_RE_HASHTAG = re.compile(r'#(\w+)')          # keep word, strip #
_RE_HTML    = re.compile(r'<[^>]+>')
_RE_EMOJI   = re.compile(
    '['
    u'\U0001F600-\U0001F64F'
    u'\U0001F300-\U0001F5FF'
    u'\U0001F680-\U0001F6FF'
    u'\U0001F1E0-\U0001F1FF'
    u'\U00002702-\U000027B0'
    u'\U000024C2-\U0001F251'
    ']+', flags=re.UNICODE)
_RE_REPEAT  = re.compile(r'(.)\1{2,}')       # 'looooove' → 'loove'
_RE_SPACE   = re.compile(r'\s+')


def clean_text(text: str) -> str:
    """Apply a reproducible cleaning pipeline to a single string."""
    text = str(text)
    text = _RE_HTML.sub(' ', text)            # strip HTML tags
    text = _RE_URL.sub('[URL]', text)         # replace URLs
    text = _RE_MENTION.sub('[USER]', text)    # replace @mentions
    text = _RE_HASHTAG.sub(r'\1', text)       # '#word' → 'word'
    text = _RE_EMOJI.sub(' ', text)           # remove emoji
    text = _RE_REPEAT.sub(r'\1\1', text)      # reduce char repetitions
    text = text.lower()                       # lowercase
    text = _RE_SPACE.sub(' ', text).strip()   # normalise whitespace
    return text


# Smoke test
sample = 'Loooove @elonmusk!! Check https://example.com #AI is 🚀🔥 <b>bold</b>'
print('Before:', sample)
print('After :', clean_text(sample))

In [ ]:
print('Cleaning text …', end=' ')
df['clean_text'] = df[TEXT_COL].apply(clean_text)

# Drop rows that became empty after cleaning
before = len(df)
df = df[df['clean_text'].str.strip() != ''].copy()
df.reset_index(drop=True, inplace=True)
print('done.')
print(f'Rows removed (empty after cleaning): {before - len(df):,}  |  Remaining: {len(df):,}')

In [ ]:
# Before / after length comparison
df['clean_char_len'] = df['clean_text'].str.len()
df['clean_word_len'] = df['clean_text'].str.split().str.len()

comparison = pd.DataFrame({
    'Raw chars'  : df['char_len'].describe().round(1),
    'Clean chars': df['clean_char_len'].describe().round(1),
    'Raw words'  : df['word_len'].describe().round(1),
    'Clean words': df['clean_word_len'].describe().round(1),
})
print('Before vs. After Cleaning')
print(comparison.to_string())

# Side-by-side sample
print('\n── Sample rows ──')
for _, row in df.sample(3, random_state=1).iterrows():
    print(f'  RAW  : {str(row[TEXT_COL])[:120]}')
    print(f'  CLEAN: {row["clean_text"][:120]}')
    print()

## Baseline Model (Logistic Regression)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# Features and target
X = df['clean_text']
y = df['status']

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert text to TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=5000)  # You can adjust max_features
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train logistic regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

# Predictions
y_pred = clf.predict(X_test_vec)

# Evaluation
print(classification_report(y_test, y_pred, target_names=le.classes_))

## BERT Transformer

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['status'])

print(df.head())
print("Label mapping:")
for class_name, class_id in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{class_id}: {class_name}")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 8

train_dataset = MentalHealthDataset(
    train_df['clean_text'],
    train_df['label'],
    tokenizer,
    max_len=MAX_LEN
)

val_dataset = MentalHealthDataset(
    val_df['clean_text'],
    val_df['label'],
    tokenizer,
    max_len=MAX_LEN
)

test_dataset = MentalHealthDataset(
    test_df['clean_text'],
    test_df['label'],
    tokenizer,
    max_len=MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
train_dataset = MentalHealthDataset(
    train_df['clean_text'],
    train_df['label'],
    tokenizer,
    max_len=MAX_LEN
)

val_dataset = MentalHealthDataset(
    val_df['clean_text'],
    val_df['label'],
    tokenizer,
    max_len=MAX_LEN
)

test_dataset = MentalHealthDataset(
    test_df['clean_text'],
    test_df['label'],
    tokenizer,
    max_len=MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=7
)

model = model.to(device)
print("Model loaded successfully")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print(class_weights)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)


In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)


In [ ]:
def train_epoch(model, data_loader, optimizer, device, loss_fn):
    model.train()

    total_loss = 0
    predictions = []
    true_labels = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(true_labels, predictions)
    macro_f1 = f1_score(true_labels, predictions, average="macro")

    return avg_loss, acc, macro_f1

In [ ]:
def eval_model(model, data_loader, device, loss_fn):
    model.eval()

    total_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = loss_fn(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(true_labels, predictions)
    macro_f1 = f1_score(true_labels, predictions, average="macro")

    return avg_loss, acc, macro_f1, predictions, true_labels

In [ ]:
EPOCHS = 2

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc, train_f1 = train_epoch(
        model, train_loader, optimizer, device, loss_fn
    )

    val_loss, val_acc, val_f1, _, _ = eval_model(
        model, val_loader, device, loss_fn
    )

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Macro F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val   Macro F1: {val_f1:.4f}")

In [ ]:
test_loss, test_acc, test_f1, test_preds, test_true = eval_model(
    model, test_loader, device, loss_fn
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

In [ ]:
class_names = list(label_encoder.classes_)
print(classification_report(test_true, test_preds, target_names=class_names))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_true, test_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()